# Population Health Analytics â€” Multi-Agent Supervisor Demo

End-to-end demo: **multi-agent supervisor** for disease prevalence analytics using:
- **Mosaic AI Agent Framework** â€” LangGraph supervisor + Genie sub-agent
- **Genie Space** â€” natural-language SQL over `demo_rwe_conditions_metric_view`
- **MLflow GenAI Evaluation** â€” synthetic eval questions scored by AI judges

### Pipeline
| Step | Description |
|------|-------------|
| 1 | `%run` the base RWE conditions setup (loads data + creates metric view) |
| 2 | Create Genie Space backed by `demo_rwe_conditions_metric_view` |
| 3 | Build multi-agent supervisor (Genie + analytics tools) |
| 4 | Log & deploy agent via MLflow / Model Serving |
| 5 | Generate synthetic evaluation questions |
| 6 | Run `mlflow.genai.evaluate()` and view results |

## Setup & Configuration

Install required packages and define catalog, schema, and model name constants used throughout the demo.

In [0]:
%pip install -U mlflow databricks-agents databricks-langchain databricks-sdk langgraph langchain langchain-core --quiet
dbutils.library.restartPython()

In [0]:
import os, json, uuid, random, hashlib
from datetime import datetime, timedelta
import mlflow
from databricks.sdk import WorkspaceClient

CATALOG = "home"
SCHEMA  = "default"

# Tables created by the run_me notebook
BASE_TABLE = "demo_rwe_conditions"
METRIC_VIEW = "demo_rwe_conditions_metric_view"
FULLY_QUALIFIED_TABLE = f"{CATALOG}.{SCHEMA}.{BASE_TABLE}"
FULLY_QUALIFIED_VIEW = f"{CATALOG}.{SCHEMA}.{METRIC_VIEW}"

# Agent config
AGENT_MODEL_NAME = f"{CATALOG}.{SCHEMA}.pop_health_agent"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

w = WorkspaceClient()
print(f"âœ“ Base table   : {FULLY_QUALIFIED_TABLE}")
print(f"âœ“ Metric view  : {FULLY_QUALIFIED_VIEW}")
print(f"âœ“ Agent model  : {AGENT_MODEL_NAME}")

## Data Foundation & Genie Space

Load the base RWE conditions dataset and metric view via `%run`, then create (or reuse) a **Genie Space** backed by `demo_rwe_conditions_metric_view` for natural-language SQL access.

In [0]:
%run ./1_setup_rwe_data

# Run this in Databricks

In [0]:
import json, uuid

GENIE_SPACE_NAME = "Population Health Analytics"
WAREHOUSE_ID = "0499d9b5566909c3"  # Serverless Starter Warehouse
genie_space_id = None

# Check for existing space
try:
    for space in w.genie.list_spaces():
        if space.title == GENIE_SPACE_NAME:
            genie_space_id = space.space_id
            print(f"\u2713 Found existing Genie space: {genie_space_id}")
            break
except Exception:
    pass

if genie_space_id is None:
    serialized_space = json.dumps({
        "version": 1,
        "config": {
            "sample_questions": [
                {"id": uuid.uuid4().hex, "question": ["What are the top 10 most common conditions?"]},
                {"id": uuid.uuid4().hex, "question": ["What is the prevalence of diabetes?"]},
                {"id": uuid.uuid4().hex, "question": ["How many patients have active hypertension?"]},
            ]
        },
        "data_sources": {
            "tables": [
                {"identifier": FULLY_QUALIFIED_VIEW}
            ]
        },
        "instructions": {
            "text_instructions": [{
                "id": uuid.uuid4().hex,
                "content": [
                    "This space contains clinical conditions data for population health analytics. "
                    "Use it to answer questions about disease prevalence, patient counts, and condition trends."
                ]
            }]
        }
    })

    genie_space = w.genie.create_space(
        warehouse_id=WAREHOUSE_ID,
        serialized_space=serialized_space,
        title=GENIE_SPACE_NAME,
        description=(
            "Population health analytics space backed by demo_rwe_conditions_metric_view. "
            "Query disease prevalence, patient conditions, and clinical event patterns."
        ),
    )
    genie_space_id = genie_space.space_id
    print(f"\u2713 Created Genie space: {genie_space_id}")

print(f"Genie Space ID: {genie_space_id}")
print(f"Backing view : {FULLY_QUALIFIED_VIEW}")

In [0]:
import time
time.sleep(20) #Takes a second to deploy Genie

In [0]:
# Look up the Genie Space ID by its display name
genie_space_id = None
for space in w.genie.list_spaces().spaces:
    if space.title == GENIE_SPACE_NAME:
        genie_space_id = space.space_id
        break

if genie_space_id is None:
    raise ValueError(f"Genie space '{GENIE_SPACE_NAME}' not found. Run the creation cell above first.")

print(f"âœ“ Genie Space: {GENIE_SPACE_NAME} â†’ {genie_space_id}")

## Multi-Agent Supervisor

Define the **LangGraph ReAct supervisor** with four tools (Genie, prevalence calculator, top conditions, population summary), log the agent to **MLflow** with Unity Catalog resources, and validate it locally before evaluation.

In [0]:
%%writefile pop_health_agent.py
import mlflow
from typing import Generator
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from databricks.connect import DatabricksSession
from databricks_langchain import ChatDatabricks
from databricks_langchain.genie import Genie
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# â”€â”€ Configuration (placeholders injected by next cell) â”€â”€
FULLY_QUALIFIED_TABLE = "__TABLE__"
FULLY_QUALIFIED_VIEW  = "__VIEW__"
GENIE_SPACE_ID        = "__GENIE_SPACE_ID__"
LLM_ENDPOINT          = "__LLM_ENDPOINT__"


def _safe_sql(val: str) -> str:
    return val.replace("'", "''")


# Initialise Spark & compute population size
spark = DatabricksSession.builder.serverless(True).getOrCreate()
num_patients = spark.sql(
    f"SELECT COUNT(DISTINCT PATIENT) FROM {FULLY_QUALIFIED_TABLE}"
).collect()[0][0]

genie = Genie(space_id=GENIE_SPACE_ID)


# Tools
@tool
def ask_genie(question: str) -> str:
    """Query the clinical conditions database using natural language via Genie.
    Use for ad-hoc data questions, exploratory analysis, or SQL-based lookups."""
    return genie.ask_question(question)


@tool
def compute_disease_prevalence(condition_description: str) -> str:
    """Compute the prevalence rate of a specific disease/condition.
    Use when asked about prevalence, rates, or percentages of a condition."""
    safe_cond = _safe_sql(condition_description)
    result = spark.sql(f"""
        SELECT DESCRIPTION,
               COUNT(*) as total_events,
               COUNT(DISTINCT PATIENT) as affected_patients,
               ROUND(COUNT(DISTINCT PATIENT) * 100.0 / {num_patients}, 2) as prevalence_pct,
               COUNT(CASE WHEN STOP IS NULL THEN 1 END) as active_cases
        FROM {FULLY_QUALIFIED_TABLE}
        WHERE LOWER(DESCRIPTION) LIKE LOWER('%' || '{safe_cond}' || '%')
        GROUP BY DESCRIPTION
    """).collect()
    if not result:
        return f"No conditions found matching '{condition_description}'."
    return "\n".join(
        f"{r['DESCRIPTION']}: {r['affected_patients']} patients "
        f"({r['prevalence_pct']}% prevalence), {r['total_events']} events, "
        f"{r['active_cases']} active" for r in result
    )


@tool
def get_top_conditions(top_n: int = 10) -> str:
    """Get the top N most prevalent conditions. Use when asked about
    most common diseases, top conditions, or disease rankings."""
    result = spark.sql(f"""
        SELECT DESCRIPTION,
               COUNT(*) as total_events,
               COUNT(DISTINCT PATIENT) as affected_patients,
               ROUND(COUNT(DISTINCT PATIENT) * 100.0 / {num_patients}, 2) as prevalence_pct
        FROM {FULLY_QUALIFIED_TABLE}
        GROUP BY DESCRIPTION ORDER BY affected_patients DESC LIMIT {top_n}
    """).collect()
    lines = [f"Top {top_n} conditions by patient count:"]
    for i, r in enumerate(result, 1):
        lines.append(f"  {i}. {r['DESCRIPTION']}: {r['affected_patients']} patients ({r['prevalence_pct']}%)")
    return "\n".join(lines)


@tool
def get_population_summary() -> str:
    """Get overall population health summary. Use for overview or general stats questions."""
    r = spark.sql(f"""
        SELECT COUNT(*) as total_events, COUNT(DISTINCT PATIENT) as total_patients,
               COUNT(DISTINCT CODE) as unique_conditions,
               COUNT(CASE WHEN STOP IS NULL THEN 1 END) as active_conditions,
               MIN(START) as earliest, MAX(START) as latest
        FROM {FULLY_QUALIFIED_TABLE}
    """).collect()[0]
    return (f"Population Health Summary:\n  Patients: {r['total_patients']}\n"
            f"  Events: {r['total_events']}\n  Unique conditions: {r['unique_conditions']}\n"
            f"  Active: {r['active_conditions']}\n  Range: {r['earliest']} - {r['latest']}")


# Build the agent graph
llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

system_prompt = f"""You are a Population Health Analytics supervisor agent.
You coordinate specialized sub-agents to answer questions about disease
prevalence, patient conditions, and clinical event patterns.

The dataset is backed by {FULLY_QUALIFIED_VIEW} with {num_patients} unique patients.

Tools:
1. **ask_genie** - Query the clinical conditions database via Genie.
2. **compute_disease_prevalence** - Calculate prevalence rates for a specific condition.
3. **get_top_conditions** - Ranked list of most prevalent conditions.
4. **get_population_summary** - Overall population health statistics.

Provide clinician-friendly answers with specific numbers and percentages.
Mention the total population size ({num_patients} patients) for context.
"""

graph = create_react_agent(
    model=llm,
    tools=[ask_genie, compute_disease_prevalence, get_top_conditions, get_population_summary],
    prompt=system_prompt,
)


# Thin ResponsesAgent adapter (required by agents.deploy)
def _to_messages(request: ResponsesAgentRequest) -> list[dict]:
    """Normalize Responses API messages to plain {role, content} dicts."""
    out = []
    for m in request.input:
        content = m.content
        if not isinstance(content, str):
            content = getattr(content[0], "text", str(content[0])) if content else ""
        out.append({"role": m.role, "content": content})
    return out


class Agent(ResponsesAgent):

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        result = graph.invoke({"messages": _to_messages(request)})
        text = next(
            (m.content for m in reversed(result["messages"])
             if getattr(m, "type", None) == "ai" and m.content
             and not getattr(m, "tool_calls", None)),
            "No response generated.",
        )
        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text=text, id="msg_1")]
        )

    def predict_stream(
        self, request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        full = ""
        for chunk, meta in graph.stream(
            {"messages": _to_messages(request)}, stream_mode="messages",
        ):
            if (
                hasattr(chunk, "content") and chunk.content
                and meta.get("langgraph_node") == "agent"
                and not getattr(chunk, "tool_calls", None)
                and not getattr(chunk, "tool_call_chunks", None)
            ):
                full += chunk.content
                yield ResponsesAgentStreamEvent(
                    **self.create_text_delta(delta=chunk.content, item_id="msg_1")
                )
        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item(text=full or "No response.", id="msg_1"),
        )


mlflow.langchain.autolog()
mlflow.models.set_model(Agent())

In [0]:
# Replace placeholders in the agent file with live notebook variables
import os

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
agent_file = os.path.join("/Workspace" + os.path.dirname(nb_path), "pop_health_agent.py")

code = open(agent_file).read()
code = (
    code
    .replace("__TABLE__", FULLY_QUALIFIED_TABLE)
    .replace("__VIEW__", FULLY_QUALIFIED_VIEW)
    .replace("__GENIE_SPACE_ID__", genie_space_id)
    .replace("__LLM_ENDPOINT__", LLM_ENDPOINT)
)
open(agent_file, "w").write(code)

print(f"\u2713 Injected config into {agent_file}")
print(f"  GENIE_SPACE_ID = {genie_space_id}")
print(f"  TABLE          = {FULLY_QUALIFIED_TABLE}")
print(f"  VIEW           = {FULLY_QUALIFIED_VIEW}")

In [0]:
import os, mlflow
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksGenieSpace,
    DatabricksSQLWarehouse,
    DatabricksTable,
)

mlflow.set_registry_uri("databricks-uc")

# Derive the notebook directory (where %%writefile placed the file)
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = "/Workspace" + os.path.dirname(nb_path)
agent_file_path = os.path.join(nb_dir, "pop_health_agent.py")
print(f"Agent file: {agent_file_path}  (exists: {os.path.isfile(agent_file_path)})")

input_example = {
    "input": [{"role": "user", "content": "What is the prevalence of diabetes in our population?"}]
}

# Declare all Databricks resources the agent depends on
resources = [
    DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-3-70b-instruct"),
    DatabricksGenieSpace(genie_space_id=genie_space_id),
    DatabricksSQLWarehouse(warehouse_id="0499d9b5566909c3"),
    DatabricksTable(table_name=FULLY_QUALIFIED_TABLE),
    DatabricksTable(table_name=FULLY_QUALIFIED_VIEW),
]

with mlflow.start_run(run_name="pop_health_supervisor_v8") as run:
    logged_model = mlflow.pyfunc.log_model(
        python_model=agent_file_path,
        name="pop_health_agent",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            "mlflow==3.6.0",
            "databricks-langchain",
            "databricks-sdk",
            "langgraph==0.3.4",
            "langchain",
            "langchain-core",
            "pydantic",
            "databricks-connect",
        ],
    )
    run_id = run.info.run_id
    print(f"\u2713 Logged to MLflow run: {run_id}")
    print(f"  Model URI: {logged_model.model_uri}")

uc_model_info = mlflow.register_model(
    model_uri=logged_model.model_uri,
    name=AGENT_MODEL_NAME,
)
print(f"\u2713 Registered: {AGENT_MODEL_NAME} v{uc_model_info.version}")

In [0]:
# Load and test the registered model from UC
loaded_agent = mlflow.pyfunc.load_model(f"models:/{AGENT_MODEL_NAME}/{uc_model_info.version}")

test_result = loaded_agent.predict(
    {"input": [{"role": "user", "content": "What are the top 5 most prevalent conditions?"}]}
)

print("\u2500\u2500 Agent Response \u2500\u2500")
# ResponsesAgent returns a dict with 'output' list
for item in test_result.get("output", []):
    for block in item.get("content", []):
        if block.get("type") == "output_text":
            print(block["text"])
print("\n\u2713 Model loaded and tested from MLflow")

## Agent Evaluation

Generate synthetic evaluation questions from the conditions data corpus, then run **Mosaic AI Evaluation** with LLM judges (`Correctness`, `RelevanceToQuery`, `Safety`, and a custom `clinical_accuracy` guideline).

In [0]:
# Generate synthetic evaluation questions for the agent

import pandas as pd
from databricks.agents.evals import generate_evals_df

# Compute session variables from the data (%%writefile doesn't set Python state)
NUM_PATIENTS = spark.sql(f"SELECT COUNT(DISTINCT PATIENT) FROM {FULLY_QUALIFIED_TABLE}").collect()[0][0]
total_events = spark.sql(f"SELECT COUNT(*) FROM {FULLY_QUALIFIED_TABLE}").collect()[0][0]

conditions_summary = spark.sql(f"""
    SELECT DESCRIPTION,
           COUNT(*) as events,
           COUNT(DISTINCT PATIENT) as patients,
           ROUND(COUNT(DISTINCT PATIENT) * 100.0 / {NUM_PATIENTS}, 2) as prevalence_pct,
           COUNT(CASE WHEN STOP IS NULL THEN 1 END) as active
    FROM {FULLY_QUALIFIED_TABLE}
    GROUP BY DESCRIPTION ORDER BY patients DESC
""").toPandas()

doc_text = f"""Population Health Analytics Dataset Summary

Backing table: {FULLY_QUALIFIED_TABLE}
Metric view: {FULLY_QUALIFIED_VIEW}

This dataset contains {total_events:,} clinical condition events across
{NUM_PATIENTS:,} patients. The data tracks disease prevalence for
population health monitoring.

Condition Prevalence Summary:
"""
for _, row in conditions_summary.iterrows():
    doc_text += (f"- {row['DESCRIPTION']}: {row['patients']} patients "
                 f"({row['prevalence_pct']}% prevalence), "
                 f"{row['events']} events, {row['active']} active\n")

docs = pd.DataFrame([{"content": doc_text, "doc_uri": FULLY_QUALIFIED_VIEW}])

agent_description = f"""
The agent is a Population Health Analytics supervisor that answers questions
about disease prevalence, patient condition patterns, and clinical event
statistics. It queries {METRIC_VIEW} via a Genie space and has analytics
tools for prevalence calculations. The population consists of {NUM_PATIENTS:,}
patients with {total_events:,} condition events.
"""

question_guidelines = """
User personas:
- A population health analyst reviewing disease burden
- A clinical informaticist exploring condition prevalence trends
- A public health officer seeking aggregate disease statistics

Example questions:
- What is the prevalence of diabetes in our population?
- Which conditions are most common?
- How many patients have active hypertension?
- Compare diabetes vs hypertension prevalence
- What percentage of patients have chronic kidney disease?

Additional guidelines:
- Questions should focus on disease prevalence, patient counts, and trends
- Include questions about specific conditions and comparative analyses
- Ask about active vs resolved conditions
- Questions should be specific and quantitative
"""

evals_df = generate_evals_df(
    docs,
    num_evals=25,
    agent_description=agent_description,
    question_guidelines=question_guidelines,
)

print(f"âœ“ Generated {len(evals_df)} synthetic evaluation questions")
display(evals_df)

In [0]:
# Run Mosaic AI Evaluation with LLM judges
# Uses the model loaded from MLflow (loaded_agent from cell 13)

import warnings
from mlflow.genai.scorers import (
    RelevanceToQuery,
    Correctness,
    Guidelines,
    Safety,
)

warnings.filterwarnings("ignore", message=".*threadpoolctl.*")

# predict_fn wraps the MLflow-loaded ResponsesAgent model
def predict_fn(messages: list[dict]) -> str:
    """Invoke the registered agent via MLflow and return the response."""
    result = loaded_agent.predict({"input": messages})
    for item in result.get("output", []):
        for block in item.get("content", []):
            if block.get("type") == "output_text":
                return block["text"]
    return "No response generated."

eval_results = mlflow.genai.evaluate(
    data=evals_df,
    predict_fn=predict_fn,
    scorers=[
        RelevanceToQuery(),
        Correctness(),
        Safety(),
        Guidelines(
            name="clinical_accuracy",
            guidelines=(
                "The response should contain specific numbers, percentages, "
                f"and patient counts. It should reference the population size "
                f"({NUM_PATIENTS:,} patients) and provide clinician-friendly language."
            ),
        ),
    ],
)

print("\u2713 Evaluation complete")
print(f"  Run ID: {eval_results.run_id}")

In [0]:
# Display evaluation results in MLflow

# View traces from the evaluation run
traces_df = mlflow.search_traces(
    run_id=eval_results.run_id,
)

print(f"Evaluation traces: {len(traces_df)} rows")
print(f"\nView full results in MLflow UI: open the experiment and select run {eval_results.run_id}")

# Display summary metrics
if hasattr(eval_results, 'metrics') and eval_results.metrics:
    print("\nâ”€â”€ Aggregate Metrics â”€â”€")
    for metric, value in eval_results.metrics.items():
        print(f"  {metric}: {value}")

# Display per-question results
print("\nâ”€â”€ Per-Question Results â”€â”€")
display(traces_df)

## Deploy to Model Serving

Deploy the evaluated agent to a **Databricks Model Serving** endpoint via the SDK.

In [0]:
from databricks import agents

# Remove stale endpoint with incompatible signature before redeploying
try:
    w.serving_endpoints.delete("pop-health-agent")
    import time; time.sleep(5)
except Exception:
    pass

deployment = agents.deploy(
    model_name=AGENT_MODEL_NAME,
    model_version=uc_model_info.version,
    endpoint_name="pop-health-agent",
)
print(f"\u2713 Deployed to endpoint: {deployment.endpoint_name}")
print(f"  Review App: {deployment.review_app_url}")